# Ruchika: Colab steps after the alpha analysis (student account)
Run the parts in order. **Part A** collects small run logs for the freeze (CPU). **Part B** scores the frozen selections (needs the GPU runtime: Runtime, Change runtime type, GPU) and must only be run AFTER the freeze is committed and tagged `selections-frozen`.
Keep outputs cleared before committing this notebook (Edit, Clear all outputs): outputs can contain dataset images.

## Part A: collect the run logs for the freeze (CPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT = "/content/drive/MyDrive/group9/checkpoints"
import os; print(sorted(os.listdir(CKPT))[:30])

In [ ]:
!rm -rf logs runs_logs.zip && mkdir -p logs/results/runs
!for d in $CKPT/ft_* $CKPT/lp_*; do n=$(basename $d); mkdir -p logs/results/runs/$n; cp $d/metrics.csv $d/run_info.json logs/results/runs/$n/; done
!ls logs/results/runs | wc -l       # should print 24
!cd logs && zip -qr ../runs_logs.zip results/runs
from google.colab import files; files.download('runs_logs.zip')

Now on your Mac: unzip `runs_logs.zip` into the repo folder (it creates `results/runs/`), run `bash scripts/06_select_and_freeze.sh results/runs`, then commit, tag `selections-frozen` and push (see `docs/runbook_ruchika.md`). Come back here for Part B.

## Part B: score the frozen selections (GPU runtime)

In [ ]:
!rm -rf repo && git clone https://github.com/RuchikaChaurasia/spectral-checkpoint-selection.git repo
%cd repo
!git tag -l                           # must list: selections-frozen
!pip install -q datasets imagehash weightwatcher pyyaml pytest scikit-learn
from google.colab import drive
drive.mount('/content/drive')
CKPT = "/content/drive/MyDrive/group9/checkpoints"
!python -m pytest -q                  # all tests pass

In [ ]:
# 1) sanity check on Product VALIDATION: the harness must agree with the training log (PASS on every line)
!python -m src.eval.sanity_check --auto --runs-dir results/runs --checkpoints-dir $CKPT

In [ ]:
# 2) Real-World + Product-test predictions for the frozen selections only (36 checkpoints; about 5 to 10 minutes)
!python -m src.eval.real_world --checkpoints-dir $CKPT

In [ ]:
# 3) OPTIONAL, descriptive: every epoch of the 12 fine-tuning runs (360 more checkpoints; about 20 to 40 minutes)
!python -m src.eval.real_world --checkpoints-dir $CKPT --all-epochs

In [ ]:
# 4) tables and figures
!python -m src.eval.aggregate
!ls results/tables results/figures

In [ ]:
# 5) package the results (small files) and download them
!zip -qr eval_results.zip results/predictions results/tables results/figures
from google.colab import files; files.download('eval_results.zip')